In [1]:
import os
import glob
from pathlib import Path
import gradio as gr 
import ollama
from dotenv import load_dotenv


In [2]:
load_dotenv(override=True)


False

In [ ]:
MODEL="qwen3:8b"


In [5]:
#load knowledge base
knowledge = {}

In [8]:
#load imployee information
filenames = glob.glob("knowledge-base/employes/*")

for filename in filenames:
    name = Path(filename).stem.split("")[-1]
    with open(filename,"r",encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()
        

In [9]:
#load prodcut information

filenames = glob.glob("knowledge-base/products/*")
for filename in filenames:
    name = Path(filename).stem 
    with open(filename,"r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()
        print (f"loaded {len(knowledge)} knowlege ducuments")
        

loaded 1 knowlege ducuments
loaded 2 knowlege ducuments
loaded 3 knowlege ducuments
loaded 4 knowlege ducuments
loaded 5 knowlege ducuments
loaded 6 knowlege ducuments
loaded 7 knowlege ducuments
loaded 8 knowlege ducuments


In [10]:
#system propmt 
SYSTEM_PREFIX = """ 
You reprsent Insurellm, the insurance Tech company.
You are an expert in ansering questions about Insurellm,
ist employess and its products.
You are provide with additional context that might be relevent to the useers question.
Give brief accurate anwsers
If you dont know the anser say so 

Relevant context:


"""

In [18]:
#retiver

def get_relevant_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())

    words = text.lower().split()
    relevant_content = [
        knowledge[word]
        for word in words 
        if word in knowledge
    ]
    return relevant_content

In [12]:
# build additinal context 
def additional_context(message):
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        return (
            "There is no additional context"
            "relevant to the user questions"
        )
    result = (
        "The following additional context might be "
        "relevant to the user questions:\n\n"
    )


    result += "\n\n".join(relevant_context)
    return result


In [13]:
#chat function
def chat (message,history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [
        {
            "role":"system",
            "content":system_message,
        }
    ]
    messages.extend(history)
    messages.append(
        {
            "role":"user",
            "content": message
        }
    )
    response = ollama.chat(
        model=MODEL,
        messages=messages
    )
    return response["message"]["content"]

In [14]:
demo = gr.ChatInterface(
    fn=chat,
)

In [20]:
demo.launch(inbrowser=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/gradio/blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/user/Downloads/llm-engineering-hands-on/venv/lib/python3.12/site-packages/gradio/blocks.py", line 1688, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/user/Downloads/llm-engineering-ha